#TASK 3 - DATA QUALITY RULES ENGINE

Goal:Evaluate Data Quality

Calculate Quality Scores

Assign Severity Levels

Generate Failed Records Report


###Import Libraries

In [1]:
import pandas as pd
import numpy as np
import sqlite3

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


###Upload Database

In [2]:
from google.colab import files
uploaded = files.upload()

Saving lab_quality.db to lab_quality.db


###Connect Database

In [3]:
conn = sqlite3.connect("lab_quality.db")
print("Connected Successfully")

Connected Successfully


###Loading Fact Table

In [4]:
fact_df = pd.read_sql(
    "SELECT * FROM fact_lab_measurements",
    conn
)

print("Shape:", fact_df.shape)

fact_df.head()

Shape: (500000, 10)


,sample_id,lab_key,instrument_key,operator_key,experiment_date,recorded_at,analyte_name,measured_value,unit,status
0,LAB-000001,1,1,1,2026-05-24 00:00:00,2026-05-24 07:00:00,Ibuprofen,107.45,g/L,Pass
1,LAB-000002,2,2,2,2025-08-27 00:00:00,2025-08-27 08:00:00,Doxycycline,97.93,ug/mL,Pass
2,LAB-000003,3,3,1,2025-07-13 00:00:00,2025-07-13 12:00:00,Diclofenac,109.72,ppm,Pass
3,LAB-000004,2,4,3,2025-11-18 00:00:00,2025-11-18 10:00:00,Diclofenac,122.85,ppm,Fail
4,LAB-000005,2,5,4,2025-11-03 00:00:00,2025-11-03 01:00:00,Doxycycline,96.49,ppm,Pending


###Date Conversion

In [5]:
fact_df["experiment_date"] = pd.to_datetime(
    fact_df["experiment_date"],
    errors="coerce"
)

fact_df["recorded_at"] = pd.to_datetime(
    fact_df["recorded_at"],
    errors="coerce"
)

print("Dates Converted")

Dates Converted


###Rule 1: sample_id NOT NULL

In [6]:
fact_df["rule_1_sample_id_not_null"] = np.where(
    fact_df["sample_id"].notna(),
    1,
    0
)

###Rule 2: experiment_date NOT NULL

In [7]:
fact_df["rule_2_experiment_date_not_null"] = np.where(
    fact_df["experiment_date"].notna(),
    1,
    0
)

###Rule 3: measured_value NOT NULL

In [8]:
fact_df["rule_3_measured_value_not_null"] = np.where(
    fact_df["measured_value"].notna(),
    1,
    0
)

###Rule 4: sample_id UNIQUE

In [9]:
duplicates = fact_df["sample_id"].duplicated(
    keep=False
)

fact_df["rule_4_sample_id_unique"] = np.where(
    duplicates,
    0,
    1
)

###Rule 5: measured_value > 0

In [10]:
fact_df["rule_5_positive_measurement"] = np.where(
    fact_df["measured_value"] > 0,
    1,
    0
)

###Rule 6: measured_value < 1000

In [11]:
fact_df["rule_6_reasonable_measurement"] = np.where(
    fact_df["measured_value"] < 1000,
    1,
    0
)

###Rule 7: VALID UNIT

In [12]:
valid_units = [
    "mg/mL",
    "ppm",
    "ug/mL",
    "g/L"
]

fact_df["rule_7_valid_unit"] = np.where(
    fact_df["unit"].isin(valid_units),
    1,
    0
)

###Rule 8: VALID STATUS

In [13]:
valid_status = [
    "Pass",
    "Fail",
    "Pending"
]

fact_df["rule_8_valid_status"] = np.where(
    fact_df["status"].isin(valid_status),
    1,
    0
)

###Rule 9: PASS CANNOT HAVE NULL VALUES

In [14]:
fact_df["rule_9_pass_requires_value"] = np.where(
    (fact_df["status"] == "Pass")
    &
    (fact_df["measured_value"].isna()),
    0,
    1
)

###Rule 10: TIMELINESS RULE (Within 24 Hours)

In [15]:
time_diff = (
    fact_df["recorded_at"]
    -
    fact_df["experiment_date"]
)

fact_df["rule_10_timely_recording"] = np.where(
    time_diff.dt.total_seconds()
    <= 86400,
    1,
    0
)

###Rule 11: instrument_id should exist

In [16]:
fact_df["rule_11_instrument_exists"] = np.where(
    fact_df["instrument_key"].notna(),
    1,
    0
)

###Rule 12: operator_id should exist

In [17]:
fact_df["rule_12_operator_exists"] = np.where(
    fact_df["operator_key"].notna(),
    1,
    0
)

###Quality Scores

In [18]:
rule_columns = [
    col
    for col in fact_df.columns
    if col.startswith("rule_")
]

fact_df["quality_score"] = (
    fact_df[rule_columns]
    .sum(axis=1)
    /
    len(rule_columns)
) * 100

fact_df["quality_score"] = (
    fact_df["quality_score"]
    .round(2)
)

fact_df[
    ["sample_id","quality_score"]
].head()

,sample_id,quality_score
0,LAB-000001,100.00
1,LAB-000002,100.00
2,LAB-000003,100.00
3,LAB-000004,100.00
4,LAB-000005,91.67


###Severity

In [19]:
fact_df["severity"] = np.select(

    [
        fact_df["quality_score"] < 50,

        (fact_df["quality_score"] >= 50)
        &
        (fact_df["quality_score"] < 80),

        fact_df["quality_score"] >= 80
    ],

    [
        "Critical",
        "Major",
        "Minor"
    ],

    default="Minor"
)

fact_df[
    ["quality_score","severity"]
].head()

,quality_score,severity
0,100.00,Minor
1,100.00,Minor
2,100.00,Minor
3,100.00,Minor
4,91.67,Minor


###Failed rule count

In [20]:
fact_df["failed_rule_count"] = (
    len(rule_columns)
    -
    fact_df[rule_columns].sum(axis=1)
)

###Overall Dataset Quality Scores

In [21]:
dataset_quality_score = round(
    fact_df["quality_score"].mean(),
    2
)

print(
    "Overall Dataset Quality Score:",
    dataset_quality_score,
    "%"
)

Overall Dataset Quality Score: 97.15 %


###Failed Records Table

In [23]:
failed_records = fact_df[
    fact_df["failed_rule_count"] > 0
]

print(
    "Failed Records:",
    len(failed_records)
)

failed_records.head()

Failed Records: 95838


,sample_id,lab_key,instrument_key,operator_key,experiment_date,recorded_at,analyte_name,measured_value,unit,status,...,rule_6_reasonable_measurement,rule_7_valid_unit,rule_8_valid_status,rule_9_pass_requires_value,rule_10_timely_recording,rule_11_instrument_exists,rule_12_operator_exists,quality_score,severity,failed_rule_count
4,LAB-000005,2,5,4,2025-11-03,2025-11-03 01:00:00,Doxycycline,96.49,ppm,Pending,...,1,1,1,1,1,1,1,91.67,Minor,1
6,LAB-000007,4,7,6,2025-09-10,2025-09-10 01:00:00,Cetirizine,NaN,ppm,Pass,...,0,1,1,0,1,1,1,66.67,Major,4
10,LAB-000011,3,6,10,2025-08-14,2025-08-14 12:00:00,Ibuprofen,NaN,ppm,Pass,...,0,1,1,0,1,1,1,66.67,Major,4
13,LAB-000014,4,7,12,2025-07-17,2025-07-17 07:00:00,Amoxicillin,-50.00,ug/mL,Pass,...,1,1,1,1,1,1,1,91.67,Minor,1
16,LAB-000017,7,2,13,2025-10-20,2025-10-20 06:00:00,Paracetamol,NaN,g/L,Fail,...,0,1,1,1,1,1,1,75.00,Major,3


###Updated Database

In [24]:
fact_df.to_sql(
    "fact_lab_measurements",
    conn,
    if_exists="replace",
    index=False
)

failed_records.to_sql(
    "failed_records",
    conn,
    if_exists="replace",
    index=False
)

print("Database Updated")

Database Updated


###Verification

In [25]:
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

tables

,name
0,dim_lab
1,dim_instrument
2,dim_operator
3,fact_lab_measurements
4,failed_records


###Save and Download

In [26]:
conn.close()

from google.colab import files

files.download(
    "lab_quality.db"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
print("Overall Dataset Quality Score:", dataset_quality_score)

Overall Dataset Quality Score: 97.15


In [28]:
fact_df["severity"].value_counts()

,count
severity,
Minor,475392
Major,24601
Critical,7
